In [1]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

train = [
    ("the movie was great", 1),
    ("i hated this movie", 0),
    ("the story was amazing", 1),
    ("the acting was excellent", 1),
    ("this movie was fantastic", 1),
    ("i really hated this", 0),
    ("the movie felt boring", 0),
    ("the story was interesting", 1),
    ("i hated the ending", 0),
    ("characters were very good", 1)
]

tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()
trainer = BpeTrainer(vocab_size=40, special_tokens=["[UNK]"])
text = [text for text, label in train]
tokenizer.train_from_iterator(text, trainer)  # trains locally in a blink+-
ids = [tokenizer.encode(sentence).ids for sentence in text]
ids

[[25, 34, 27, 7, 32, 1, 16],
 [9, 29, 39, 38, 34],
 [25, 37, 36, 20, 27, 1, 11, 1, 21, 28],
 [25, 1, 3, 16, 28, 27, 5, 19, 3, 5, 10, 10, 5, 12, 16],
 [38, 34, 27, 6, 1, 12, 16, 23, 16, 9, 3],
 [9, 32, 1, 10, 10, 20, 29, 39, 38],
 [25, 34, 6, 5, 10, 16, 2, 36, 28],
 [25, 37, 36, 20, 27, 24, 26, 32, 37, 28],
 [9, 29, 39, 25, 5, 12, 4, 28],
 [3, 29, 14, 1, 3, 26, 14, 15, 18, 5, 32, 17, 5, 14, 20, 7, 13, 13, 4]]

In [2]:
tokenizer.encode("the movie was amazing").id


AttributeError: 'tokenizers.Encoding' object has no attribute 'id'

In [3]:
words = sorted({w for text, label in train for w in text.split()})
stoi = {w: i for i, w in enumerate(words)}

In [4]:
import torch
import torch.nn as nn


class ProffessionaSentimentNet(nn.Module):
    def __init__(self, stoi , dim = 8 , hidden_dim = 16):
        super().__init__()
        self.stoi = stoi
        self.emb = nn.Embedding(num_embeddings=len(stoi) , embedding_dim=dim)


        self.classifier = nn.Sequential(
            nn.Linear(dim , hidden_dim),#dropout works for this layer
            nn.ReLU(),
            nn.Dropout(p = 0.5), #randomly turns off 50% neuron to contorl overfitting
            nn.Linear(hidden_dim , 2),
        )
    def forward(self , ids):

        pooled = self.emb(ids).mean(dim = 0 , keep_dim = True)
        return self.classifier(pooled) 

# Full rag pipeline


In [2]:

import chromadb
from chunk_and_embed import chunks, chunk_embeddings, embed

# 1. Create a persistent local database
client = chromadb.PersistentClient(path="./rag_db")

# 2. Get or create a collection
collection = client.get_or_create_collection(
    name="ml_knowledge",
    metadata={"hnsw:space": "cosine"}
)

# 3. Store chunks, their vectors, and metadata
collection.upsert(
    ids=[f"chunk_{i}" for i in range(len(chunks))],
    documents=chunks,
    embeddings=chunk_embeddings.tolist(),
    metadatas=[
        {"chunk_index": i, "source": "sample_document"}
        for i in range(len(chunks))
    ]
)

print("Stored chunks:", collection.count())


# 4. Retrieve the most relevant chunks for a question
def retrieve(question, top_k=3):
    query_vector = embed(question)

    results = collection.query(
        query_embeddings=[query_vector.tolist()],
        n_results=min(top_k, collection.count()),
        include=["documents", "metadatas", "distances"]
    )

    retrieved = []

    for document, metadata, distance in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        retrieved.append({
            "text": document,
            "metadata": metadata,
            "distance": distance
        })

    return retrieved


# 5. Test retrieval
if __name__ == "__main__":
    question = "How does early stopping help prevent overfitting?"

    print("\nQuestion:", question)
    print("\nRetrieved results:")

    for rank, result in enumerate(retrieve(question), start=1):
        print(f"\nRank: {rank}")
        print("Chunk:", result["metadata"]["chunk_index"])
        print("Cosine distance:", round(result["distance"], 4))
        print("Text:", result["text"])


Stored chunks: 13

Question: How does early stopping help prevent overfitting?

Retrieved results:

Rank: 1
Chunk: 2
Cosine distance: 0.5939
Text: is a validation set and techniques like early stopping. A random forest

Rank: 2
Chunk: 0
Cosine distance: 0.736
Text: Overfitting is when a model memorises its training data and fails on

Rank: 3
Chunk: 3
Cosine distance: 0.8268
Text: stopping. A random forest averages many decision trees to reduce those errors.
